# German Traffic Sign Recognition

In [1]:
import os
import cv2
import joblib
import numpy as np
import pandas as pd
from PIL import Image
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras.applications import VGG19
from tensorflow.keras.layers import Dense, Flatten
from tensorflow.keras.models import Model

In [2]:
data_dir = r"C:\Users\CASPER\Documents\AI-YAPAY ZEKA\ödevler\Become a Pro\german traffic sign recognition"
train_dir = os.path.join(data_dir, "train")

In [3]:
IMG_HEIGHT = 32
IMG_WIDTH = 32

In [4]:
data = []
labels = []

In [5]:
if os.path.exists(train_dir):
  categories = sorted(os.listdir(train_dir))
  for category in categories:
    cat_path = os.path.join(train_dir, category)
    if os.path.isdir(cat_path):
      for img_name in os.listdir(cat_path):
        img_path = os.path.join(cat_path, img_name)
        try:
          image = Image.open(img_path)
          image = image.resize((IMG_HEIGHT, IMG_WIDTH))
          data.append(np.array(image))
          labels.append(int(category))
        except Exception as e:
          continue

In [6]:
X = np.array(data) / 255.0  
y = np.array(labels)

In [7]:
x_train, x_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [8]:
NUM_CATEGORIES = len(np.unique(y))

In [9]:
base_model = VGG19(
    input_shape=(IMG_HEIGHT, IMG_WIDTH, 3), weights="imagenet", include_top=False
)

80134624/80134624 ━━━━━━━━━━━━━━━━━━━━ 59s 1us/step


In [10]:
for layer in base_model.layers:
  layer.trainable = False

In [11]:
x = Flatten()(base_model.output)
prediction = Dense(NUM_CATEGORIES, activation="softmax")(x)

In [12]:
model = Model(inputs=base_model.input, outputs=prediction)

In [13]:
model.compile(
    loss="sparse_categorical_crossentropy",
    optimizer="adam",
    metrics=["accuracy"],
)

In [14]:
history = model.fit(
    x_train,
    y_train,
    validation_data=(x_test, y_test),
    epochs=5,
    batch_size=32,
)

Epoch 1/5
981/981 ━━━━━━━━━━━━━━━━━━━━ 512s 519ms/step - accuracy: 0.3743 - loss: 2.2239 - val_accuracy: 0.4846 - val_loss: 1.8184
Epoch 2/5
981/981 ━━━━━━━━━━━━━━━━━━━━ 498s 507ms/step - accuracy: 0.5220 - loss: 1.6745 - val_accuracy: 0.5400 - val_loss: 1.5789
Epoch 3/5
981/981 ━━━━━━━━━━━━━━━━━━━━ 501s 506ms/step - accuracy: 0.5749 - loss: 1.4808 - val_accuracy: 0.5765 - val_loss: 1.4394
Epoch 4/5
981/981 ━━━━━━━━━━━━━━━━━━━━ 537s 542ms/step - accuracy: 0.6075 - loss: 1.3587 - val_accuracy: 0.6071 - val_loss: 1.3412
Epoch 5/5
981/981 ━━━━━━━━━━━━━━━━━━━━ 527s 537ms/step - accuracy: 0.6332 - loss: 1.2714 - val_accuracy: 0.6301 - val_loss: 1.2712


In [15]:
model.save("traffic_sign_model.h5")